<a href="https://colab.research.google.com/github/shayanR10/FIV1/blob/prod/notebooks/FIV1_monolith.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FIV1 Modular Pipeline

This notebook now orchestrates the project through the reusable Python package in `src/`.

In [ ]:
from pathlib import Path

from src.scraper import build_species_list, run_harvest_pipeline
from src.filter import build_clip_filter, semantic_clear_species_images
from src.fallback import fetch_dinosaur_taxonomy, classify_with_confidence
from src.model import train_convnext


In [ ]:
KAGGLE_USERNAME = "<your-kaggle-username>"
DATA_DIR = Path("./fossil-data")
MAX_SPECIES = 40
MAX_IMAGES_PER_SPECIES = 200

species_list = build_species_list(limit=MAX_SPECIES, kaggle_username=KAGGLE_USERNAME)
clip_filter = build_clip_filter()

summary = run_harvest_pipeline(
    species_list=species_list,
    output_dir=DATA_DIR,
    kaggle_username=KAGGLE_USERNAME,
    max_images_per_species=MAX_IMAGES_PER_SPECIES,
    post_process=lambda species, directory: semantic_clear_species_images(
        species=species, data_dir=directory, clip_filter=clip_filter
    ),
)
summary


In [ ]:
taxonomy_index = fetch_dinosaur_taxonomy()

prediction = classify_with_confidence(
    species_name="Ouranosaurus",
    confidence_score=0.62,
    taxon_id="67",
    taxonomy_index=taxonomy_index,
)
prediction


In [ ]:
# Train ConvNeXt after images are available in DATA_DIR
results = train_convnext(dataset_path=DATA_DIR)
results
